# Circuit-breaker ablation (R0–R6), Qwen2-VL-2B on a T4

To pick up code changes later: `!git pull` in the repo folder.

Order: setup → smoke checks → baseline eval → R0…R5 → pick best → R6 → table + curves.
Everything is written to Google Drive (`out/` is a symlink) so a disconnect loses nothing.
Each run takes about 10–15 min to train plus 15–25 min to evaluate on a T4.

In [ ]:
!nvidia-smi
!pip -q install -U transformers accelerate peft qwen-vl-utils pillow matplotlib

In [ ]:
# Code from GitHub; data (manifests + images) from the zip built on the Mac by src/pack_for_colab.py
%cd /content
!git clone https://github.com/aku134/vlm-representation-safety.git || (cd vlm-representation-safety && git pull)
%cd /content/vlm-representation-safety
from google.colab import files
files.upload()   # data/colab_upload.zip
!unzip -q -o colab_upload.zip 'data/*' 'images/*' && git checkout -- src scripts 2>/dev/null; ls src scripts data

In [ ]:
# Persist all runs on Drive
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/mmcb_out && rm -rf out && ln -s /content/drive/MyDrive/mmcb_out out && ls out

In [ ]:
# Llama-Guard-3-1B is gated: accept the license on its HF page, then log in
from huggingface_hub import login
login()

## Smoke checks (Tasks 0–5): about 3 min each. Check that each run folder is complete and the logged losses look sane

In [ ]:
!python src/eval_cb.py --run-dir out/runs/base_smoke --base-only --smoke && python src/judge.py --run-dir out/runs/base_smoke
!bash scripts/run_experiment.sh smoke_t0 --smoke                                   # Task 0: full folder
!bash scripts/run_experiment.sh smoke_t2 --smoke --retain-layers all               # Task 2: retain_loss_raw > 0
!bash scripts/run_experiment.sh smoke_t3 --smoke --retain-layers all --kl-coeff 1  # Task 3: kl_raw ~0, small
!bash scripts/run_experiment.sh smoke_t4 --smoke --retain-layers all --kl-coeff 1 --normalize-losses  # Task 4: *_norm ~1 at step 1
!bash scripts/run_experiment.sh smoke_t5 --smoke --retain-layers all --kl-coeff 1 --normalize-losses --target-layers 18,22  # Task 5: peak_mem_gb
!ls out/runs/smoke_t0 out/runs/smoke_t0/eval; cat out/runs/smoke_t*/FAILED.txt 2>/dev/null
!python scripts/compare_runs.py out/runs/base_smoke out/runs/smoke_t*

In [ ]:
import json, glob
for f in sorted(glob.glob('out/runs/smoke_t*/train_log.json')):
    L = json.load(open(f))['log']
    print(f, '\n  first:', L[0], '\n  last: ', L[-1])

## Baseline eval (Task 1): 50 harmful, 46 benign (all the benign the eval split has)

In [ ]:
!python src/eval_cb.py --run-dir out/runs/base --base-only && python src/judge.py --run-dir out/runs/base

## Ablation: one change per run. 200 steps, lr 1e-4, seed 0, RMU unless noted

In [ ]:
!bash scripts/run_experiment.sh R0 --target-layers 9,18 --retain-layers targets

In [ ]:
!bash scripts/run_experiment.sh R1 --target-layers 9,18 --retain-layers all

In [ ]:
!bash scripts/run_experiment.sh R2 --target-layers 9,18 --retain-layers all --kl-coeff 1.0

In [ ]:
!bash scripts/run_experiment.sh R3 --target-layers 9,18 --retain-layers all --kl-coeff 1.0 --normalize-losses

In [ ]:
!bash scripts/run_experiment.sh R4 --target-layers 18,22 --retain-layers all --kl-coeff 1.0 --normalize-losses

In [ ]:
# R5 needs data/manifest_train_pairs.json (Task 6, after the pairs are reviewed)
!bash scripts/run_experiment.sh R5 --target-layers 18,22 --retain-layers all --kl-coeff 1.0 --normalize-losses --manifest data/manifest_train_pairs.json

In [ ]:
!python scripts/compare_runs.py out/runs/base out/runs/R0 out/runs/R1 out/runs/R2 out/runs/R3 out/runs/R4 out/runs/R5

In [ ]:
# R6 = best run's exact flags with --cb-loss cosine. EDIT after reading the table:
BEST_FLAGS = "--target-layers 18,22 --retain-layers all --kl-coeff 1.0 --normalize-losses"
!bash scripts/run_experiment.sh R6 {BEST_FLAGS} --cb-loss cosine

In [ ]:
!mkdir -p results
!python scripts/compare_runs.py out/runs/base out/runs/R0 out/runs/R1 out/runs/R2 out/runs/R3 out/runs/R4 out/runs/R5 out/runs/R6 --out results/cb_ablation.md
BEST = 'R3'  # EDIT
!python scripts/plot_curves.py out/runs/R0 out/runs/{BEST} --out results/cb_curves.png
!zip -r -q cb_results.zip results out/runs/*/args.json out/runs/*/train_log.json out/runs/*/eval_summary.json out/runs/*/eval out/runs/*/FAILED.txt
files.download('cb_results.zip')